# Customer Retention Analytics — Exploratory Analysis & Modeling

End-to-end walkthrough: load data, explore churn patterns, train and compare models, and render the chart set used in the README / Power BI dashboard.


In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)


## 1. Load data


In [ ]:
df = pd.read_csv('../data/customer_retention.csv')
df['target'] = (df['churn'] == 'Yes').astype(int)
df.head()


In [ ]:
df.describe(include='all').T


## 2. Churn overview


In [ ]:
churn_rate = df['target'].mean()
print(f'Overall churn rate: {churn_rate:.1%}')

fig, ax = plt.subplots(figsize=(4,4))
df['churn'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=ax, colors=['#2f6690', '#d1495b'])
ax.set_ylabel('')
ax.set_title('Churn Distribution')
plt.show()


## 3. Churn by contract type and tenure


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.barplot(data=df, x='contract_type', y='target', ax=axes[0], color='#2f6690')
axes[0].set_title('Churn Rate by Contract Type')
axes[0].set_ylabel('Churn Rate')

sns.histplot(data=df, x='tenure_months', hue='churn', bins=30, ax=axes[1], multiple='stack')
axes[1].set_title('Tenure Distribution by Churn')

plt.tight_layout()
plt.show()


## 4. Correlations among numeric features


In [ ]:
numeric_cols = ['tenure_months', 'monthly_charges', 'total_charges', 'support_tickets',
                'avg_monthly_usage_gb', 'num_products', 'satisfaction_score', 'target']

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()


## 5. Train and compare models

The full training pipeline (preprocessing, model comparison, scoring, segment summary, and chart export) lives in `src/retention_model.py` so it can be run identically from the CLI, CI, or Docker. This cell just invokes it.


In [ ]:
!python ../src/retention_model.py


## 6. Review generated outputs


In [ ]:
predictions = pd.read_csv('../outputs/customer_retention_predictions.csv')
feature_importance = pd.read_csv('../outputs/feature_importance.csv')
model_comparison = pd.read_csv('../outputs/model_comparison.csv')

model_comparison


In [ ]:
feature_importance.head(10)


## 7. Generated charts

Saved to `outputs/images/` and used directly in `README.md`:

- `roc_curve.png`
- `confusion_matrix.png`
- `feature_importance.png`
- `churn_by_segment.png`
- `model_comparison.png`


## Next steps

- Publish the Power BI dashboard (see `POWERBI.md`) and link it from the README
- Deploy the Streamlit app (`app.py`) for a live, interactive demo
- Retrain periodically as new customer data becomes available
